# Chunk evaluation

Read the published Notebook 05 handoff in a fresh project kernel. Notebook 06 starts with
the two existing paragraph plans, measured on the same parents. Counts and limits in this
initial comparison use Unicode code points. The analysis frame carries source, format, year,
length, grouping, and the inherited overlap evidence. Known overlap records have a separate
diagnostic view.

In [ ]:
import hashlib
import json
import os
import re
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "configs/preprocessing_notebook.json").is_file()
    ),
    None,
)
if ROOT is None or Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Open the project workspace and select its registered kernel.")

config = json.loads((ROOT / "configs/preprocessing_notebook.json").read_text())
profile = os.environ.get("FOT06_PROFILE", config["default_profile"])
resource = config["profiles"][profile]
output_root = (ROOT / config["output_directory"]).resolve()
HANDOFF = Path(os.environ.get("FOT06_INPUT", output_root / "formal_layout_step_v1_handoff_v1"))
HANDOFF = HANDOFF.resolve()
if HANDOFF.parent != output_root:
    raise ValueError("Select a versioned Notebook 05 handoff under the preprocessing root.")

if SparkSession.getActiveSession() is not None:
    raise RuntimeError("Start Notebook 06 in a fresh kernel.")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS", f"--driver-memory {resource['driver_memory']} pyspark-shell"
)
spark = (
    SparkSession.builder.master(resource["master"])
    .appName("fear-temperature-chunk-evaluation")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", resource["shuffle_partitions"])
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024**2))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
if spark.version != config["spark_version"] or spark.sparkContext.master != resource["master"]:
    raise RuntimeError("Spark runtime differs from the selected project profile.")
print(f"Spark {spark.version} · {spark.sparkContext.master} · Notebook 06")

## Verify the input and define the comparison frame

Verify the manifest, closure receipt and Parquet file inventory. The release group is the
sampling unit for later boundary review. Keep the comparison and overlap diagnostic cohorts
available separately. Status without a match refers to the scope of the bound cross-split
audit.

In [ ]:
def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


handoff_manifest_path = HANDOFF / "MANIFEST.json"
closure_path = HANDOFF / "05_CLOSURE.json"
manifest = json.loads(handoff_manifest_path.read_text())
closure = json.loads(closure_path.read_text())
if (
    manifest["schema_version"] != 1
    or sha256_file(closure_path) != manifest["audit_sha256"]
    or closure["status"] != "verified_text_and_baseline_plans_for_chunk_evaluation"
):
    raise ValueError("Notebook 06 requires a verified Notebook 05 text handoff.")

expected = closure["checks"]["parent_rows"]
targets = manifest["targets_codepoints"]
if (
    expected != 718_330
    or manifest["targets_codepoints"] != [1200, 2400]
    or any(
        closure["checks"][field] != expected
        for field in (
            "unique_release_rows",
            "unique_native_identities",
            "verified_body_hashes",
            "parents_with_valid_plans",
            "parents_with_valid_paragraphs",
        )
    )
    or closure["step1_new_cross_split_exact_pairs"] != 0
    or closure["step2_new_cross_split_exact_pairs"] != 0
):
    raise ValueError("The 05 handoff has unresolved roster or baseline coverage checks.")

records_dir = HANDOFF / "records"
parts = manifest["tables"]["records"]
inventory = {path.relative_to(HANDOFF).as_posix(): path for path in records_dir.glob("*.parquet")}
if (
    not (records_dir / "_SUCCESS").is_file()
    or set(inventory) != {part["path"] for part in parts}
    or sum(part["rows"] for part in parts) != expected
):
    raise ValueError("The handoff Parquet inventory is incomplete.")
for part in parts:
    file = inventory[part["path"]]
    if (
        sha256_file(file) != part["sha256"]
        or pq.ParquetFile(file).metadata.num_rows != part["rows"]
    ):
        raise ValueError(f"The handoff Parquet part failed verification: {part['path']}")

parents = spark.read.parquet(str(records_dir))
strata = parents.select(
    "release_row",
    "native_identity",
    "release_group_id",
    "ml_split",
    "source_id",
    "source_host",
    "format",
    "publication_year",
    "publication_precision",
    "clean_sha256",
    "clean_chars",
    "overlap_status",
    "overlap_pair_ids",
    "cross_split_review_required",
    "plans",
).withColumn(
    "length_band",
    F.when(F.col("clean_chars") <= 250, "00001-00250")
    .when(F.col("clean_chars") <= 1200, "00251-01200")
    .when(F.col("clean_chars") <= 2400, "01201-02400")
    .when(F.col("clean_chars") <= 5000, "02401-05000")
    .when(F.col("clean_chars") <= 10000, "05001-10000")
    .otherwise("10001+"),
)
comparison_parents = strata.where(~F.col("cross_split_review_required"))
overlap_diagnostics = strata.where(F.col("cross_split_review_required"))
counts = [
    row.asDict()
    for row in strata.groupBy("ml_split", "overlap_status")
    .count()
    .orderBy("ml_split", "overlap_status")
    .collect()
]
if counts != closure["overlap_status_by_split"]:
    raise ValueError("The evaluation strata disagree with the bound overlap counts.")

group_conflicts = (
    strata.groupBy("release_group_id")
    .agg(F.countDistinct("ml_split").alias("split_count"))
    .where(F.col("split_count") > 1)
    .limit(1)
    .count()
)
if group_conflicts or strata.where(F.col("release_group_id").isNull()).limit(1).count():
    raise ValueError("A release group is missing or crosses the inherited split.")

print(f"Verified {expected:,} parents and {len(parts)} handoff files.")
print(f"Notebook 05 handoff manifest SHA-256: {sha256_file(handoff_manifest_path)}")
display(pd.DataFrame(counts))

## Measure the existing paragraph plans

Measure the 1,200 and 2,400 code-point baselines, including single-chunk rates and chunks that
exceed the target. The `baseline_by_stratum` view keeps the format, length and overlap cohorts
available for the next comparison.

In [ ]:
plan_rows = (
    strata.select(
        "release_row",
        "format",
        "length_band",
        "cross_split_review_required",
        F.explode("plans").alias("plan"),
    )
    .select(
        "release_row",
        "format",
        "length_band",
        "cross_split_review_required",
        F.col("plan.target").alias("target"),
        F.col("plan.ends").alias("ends"),
    )
    .withColumn(
        "chunk_lengths",
        F.zip_with(
            "ends",
            F.slice(F.concat(F.array(F.lit(0)), "ends"), 1, F.size("ends")),
            lambda end, start: end - start,
        ),
    )
    .withColumn("chunks", F.size("ends"))
    .withColumn(
        "oversized_chunks", F.size(F.filter("chunk_lengths", lambda n: n > F.col("target")))
    )
)
baseline_by_stratum = plan_rows.groupBy(
    "target", "format", "length_band", "cross_split_review_required"
).agg(
    F.count("*").alias("parents"),
    F.sum("chunks").alias("chunks"),
    F.count_if(F.col("chunks") == 1).alias("one_chunk_parents"),
    F.sum("oversized_chunks").alias("chunks_over_target"),
    F.count_if(F.col("oversized_chunks") > 0).alias("parents_with_chunk_over_target"),
)
baseline_totals = (
    baseline_by_stratum.groupBy("target")
    .agg(
        F.sum("parents").alias("parents"),
        F.sum("chunks").alias("chunks"),
        F.sum("one_chunk_parents").alias("one_chunk_parents"),
        F.sum("chunks_over_target").alias("chunks_over_target"),
        F.sum("parents_with_chunk_over_target").alias("parents_with_chunk_over_target"),
    )
    .orderBy("target")
    .collect()
)
if (
    [row.target for row in baseline_totals] != targets
    or any(row.parents != expected for row in baseline_totals)
    or [{"target": row.target, "chunks": row.chunks} for row in baseline_totals]
    != closure["final_baseline_chunks"]
):
    raise ValueError("The measured baselines do not match the verified handoff.")
display(pd.DataFrame([row.asDict() for row in baseline_totals]))
print("Measured both existing code-point plans on the identical parent roster.")

## Check exact chunk reuse across splits

Hash exact slices of the verified text. Report distinct hash buckets spanning two or three
inherited splits and the total number of occurrences in those buckets. A chunk can match
across splits even when its complete parent body is distinct. Report all lengths and a
separate diagnostic at 500 code points. The comparison columns recompute the counts with
the known parent-overlap diagnostic cohort held aside. Repeated lexical content and shared boilerplate can
both produce a match. The hash buckets define a review set.

In [ ]:
chunk_occurrences = (
    parents.select(
        "release_row",
        "ml_split",
        "cross_split_review_required",
        "clean_text",
        F.explode("plans").alias("plan"),
    )
    .select(
        "release_row",
        "ml_split",
        "cross_split_review_required",
        "clean_text",
        F.col("plan.target").alias("target"),
        F.col("plan.ends").alias("ends"),
        F.posexplode("plan.ends").alias("ordinal", "end"),
    )
    .withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
    .withColumn("chunk_codepoints", F.col("end") - F.col("start"))
    .withColumn(
        "chunk_sha256",
        F.sha2(F.substring("clean_text", F.col("start") + 1, F.col("chunk_codepoints")), 256),
    )
    .drop("clean_text", "ends")
)
# Each bit represents one inherited split. A mask above one bit spans splits.
split_bit = (
    F.when(F.col("ml_split") == "train", 1).when(F.col("ml_split") == "validation", 2).otherwise(4)
)
in_comparison = ~F.col("cross_split_review_required")
cross_split_chunk_reuse = (
    chunk_occurrences.groupBy("target", "chunk_sha256", "chunk_codepoints")
    .agg(
        F.bit_or(split_bit).alias("split_mask"),
        F.bit_or(F.when(in_comparison, split_bit).otherwise(0)).alias("comparison_split_mask"),
        F.count("*").alias("chunk_occurrences"),
        F.count_if(in_comparison).alias("comparison_occurrences"),
    )
    .where(F.col("split_mask").isin(3, 5, 6, 7))
)
crosses_comparison = F.col("comparison_split_mask").isin(3, 5, 6, 7)
reuse_counts = (
    cross_split_chunk_reuse.withColumn(
        "minimum_codepoints",
        F.explode(
            F.when(F.col("chunk_codepoints") >= 500, F.array(F.lit(0), F.lit(500))).otherwise(
                F.array(F.lit(0))
            )
        ),
    )
    .groupBy("target", "minimum_codepoints")
    .agg(
        F.count("*").alias("cross_split_hash_buckets"),
        F.sum("chunk_occurrences").alias("chunk_occurrences"),
        F.count_if(crosses_comparison).alias("comparison_hash_buckets"),
        F.sum(F.when(crosses_comparison, F.col("comparison_occurrences")).otherwise(0)).alias(
            "comparison_chunk_occurrences"
        ),
    )
)
reuse_report = (
    spark.createDataFrame(
        [(target, cutoff) for target in targets for cutoff in (0, 500)],
        ["target", "minimum_codepoints"],
    )
    .join(reuse_counts, ["target", "minimum_codepoints"], "left")
    .fillna(0)
    .orderBy("target", "minimum_codepoints")
    .collect()
)
display(pd.DataFrame([row.asDict() for row in reuse_report]))
print("Measured exact chunk reuse across splits by baseline and length.")

## Prepare the full review queue for complex text

Queue every parent in the declared publication formats or designated source IDs. Include
long bodies and parents with oversized chunks, retained formatting markers, upstream review
evidence, or known overlap in any format. Record every reason. The source IDs in
`FOT06_COMPLEX_SOURCE_IDS` extend the scope to complete sources. Pin the scope and input
version before starting review. Set `FOT06_REVIEW_OUTPUT` to write an alternate scope.

The queue binds the verified body and its complete 1,200-code-point baseline plan. Each
queued parent receives consecutive review windows covering the whole body. Neighbouring
context accompanies each window. Review window boundaries only assign work and carry no
authority as chunk boundaries. This snapshot measures the review workload. A later
tokenizer-aware plan requires its own bound review packets.


In [ ]:
COMPLEX_FORMATS = (
    "Article",
    "Blog",
    "Community publication",
    "Literary",
    "News",
    "Newsletter",
    "Other",
    "Public submission",
    "Transcript",
    "Unresolved",
)
DESIGNATED_SOURCE_IDS = sorted(
    {
        source.strip()
        for source in os.environ.get("FOT06_COMPLEX_SOURCE_IDS", "").split(",")
        if source.strip()
    }
)
REVIEW_CORE_CODEPOINTS = 4_800
REVIEW_CONTEXT_CODEPOINTS = 320
REVIEW_BASELINE_TARGET = 1_200
REVIEW_POLICY = "complex_boundary_review_v1"
near_delta_path = HANDOFF.with_name(f"{HANDOFF.name}_near_delta_v1")
near_manifest_path = near_delta_path / "MANIFEST.json"
near_manifest = json.loads(near_manifest_path.read_text())
near_audit_path = near_delta_path / "AUDIT.json"
if near_manifest["input_handoff_manifest_sha256"] != sha256_file(
    handoff_manifest_path
) or near_manifest["audit_sha256"] != sha256_file(near_audit_path):
    raise ValueError("The incremental near-duplicate audit does not match this handoff.")


def bound_review_table(root, name, entries):
    directory = root / name
    inventory = {p.relative_to(root).as_posix(): p for p in directory.glob("*.parquet")}
    if not (directory / "_SUCCESS").is_file() or set(inventory) != {
        entry["path"] for entry in entries
    }:
        raise ValueError(f"Incomplete review table {name}.")
    for entry in entries:
        p = inventory[entry["path"]]
        if (
            sha256_file(p) != entry["sha256"]
            or pq.ParquetFile(p).metadata.num_rows != entry["rows"]
        ):
            raise ValueError(f"The review table failed verification: {entry['path']}")
    return spark.read.parquet(str(directory))


near_pairs = bound_review_table(
    near_delta_path, "cross_split_candidates", near_manifest["tables"]["cross_split_candidates"]
)
near_parents = (
    near_pairs.where(F.col("after.match"))
    .select(F.explode(F.array("row_a", "row_b")).alias("release_row"))
    .distinct()
    .withColumn("near_delta_match", F.lit(True))
)
default_queue_dir = HANDOFF.with_name(f"{HANDOFF.name}_chunk_review_v1")
queue_dir = Path(os.environ.get("FOT06_REVIEW_OUTPUT", default_queue_dir)).resolve()
if queue_dir.parent != HANDOFF.parent or queue_dir in (HANDOFF, near_delta_path):
    raise ValueError("Select a separate queue directory under the preprocessing root.")
queue_contract = {
    "review_policy": REVIEW_POLICY,
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
    "near_delta_manifest_sha256": sha256_file(near_manifest_path),
    "complex_formats": list(COMPLEX_FORMATS),
    "designated_source_ids": DESIGNATED_SOURCE_IDS,
    "baseline_target_codepoints": REVIEW_BASELINE_TARGET,
    "review_core_codepoints": REVIEW_CORE_CODEPOINTS,
    "review_context_codepoints": REVIEW_CONTEXT_CODEPOINTS,
}

if queue_dir.exists():
    queue_manifest = json.loads((queue_dir / "MANIFEST.json").read_text())
    queue_audit_path = queue_dir / "AUDIT.json"
    if any(
        queue_manifest.get(key) != value for key, value in queue_contract.items()
    ) or queue_manifest["audit_sha256"] != sha256_file(queue_audit_path):
        raise ValueError("Use a new review version for changed sources, inputs or review rules.")
    review_queue = bound_review_table(queue_dir, "queue", queue_manifest["queue_files"])
    queue_audit = json.loads(queue_audit_path.read_text())
else:
    if DESIGNATED_SOURCE_IDS:
        actual_ids = {
            row.source_id
            for row in parents.where(F.col("source_id").isin(*DESIGNATED_SOURCE_IDS))
            .select("source_id")
            .distinct()
            .collect()
        }
        if actual_ids != set(DESIGNATED_SOURCE_IDS):
            raise ValueError("The designated review sources include IDs outside the formal roster.")

    baseline = F.try_element_at(
        F.filter("plans", lambda plan: plan.target == REVIEW_BASELINE_TARGET), F.lit(1)
    )
    review_population = (
        parents.select(
            "release_row",
            "native_identity",
            "release_group_id",
            "ml_split",
            "format",
            "source_id",
            "source_host",
            "clean_sha256",
            "clean_chars",
            "overlap_status",
            "cross_split_review_required",
            "retained_marker_counts",
            "structure_review_reasons",
            "first_pass_review_reasons",
            "layout_review_reasons",
            "plans",
        )
        .join(F.broadcast(near_parents), "release_row", "left")
        .withColumn("plan", baseline)
    )
    long_chunk = F.exists(
        F.zip_with(
            "plan.ends",
            F.slice(F.concat(F.array(F.lit(0)), "plan.ends"), 1, F.size("plan.ends")),
            lambda end, start: end - start,
        ),
        lambda chars: chars > REVIEW_BASELINE_TARGET,
    )
    reasons = {
        "complex_publication_format": F.col("format").isin(*COMPLEX_FORMATS),
        "designated_source": (
            F.col("source_id").isin(*DESIGNATED_SOURCE_IDS)
            if DESIGNATED_SOURCE_IDS
            else F.lit(False)
        ),
        "long_body": F.col("clean_chars") > 2400,
        "oversized_baseline_chunk": long_chunk,
        "retained_markup_or_spacing": F.size("retained_marker_counts") > 0,
        "upstream_review_evidence": (
            (F.size("structure_review_reasons") > 0)
            | (F.size("first_pass_review_reasons") > 0)
            | (F.size("layout_review_reasons") > 0)
        ),
        "known_parent_overlap": F.col("cross_split_review_required"),
        "incremental_near_duplicate_match": F.col("near_delta_match"),
    }
    review_population = review_population.withColumn(
        "review_reasons",
        F.filter(
            F.array(*[F.when(condition, F.lit(name)) for name, condition in reasons.items()]),
            lambda reason: reason.isNotNull(),
        ),
    )
    queue_fields = (
        "release_row",
        "native_identity",
        "release_group_id",
        "ml_split",
        "format",
        "source_id",
        "source_host",
        "clean_sha256",
        "clean_chars",
        "overlap_status",
        "review_reasons",
        "plan",
    )
    cores = F.sequence(F.lit(0), F.col("clean_chars") - 1, F.lit(REVIEW_CORE_CODEPOINTS))
    queue_source = (
        review_population.where(F.size("review_reasons") > 0)
        .select(*queue_fields)
        .withColumn("plan_sha256", F.sha2(F.to_json("plan"), 256))
        .withColumn(
            "review_units",
            F.transform(
                cores,
                lambda start, ordinal: F.struct(
                    ordinal.alias("unit_ordinal"),
                    start.alias("start"),
                    F.least(start + REVIEW_CORE_CODEPOINTS, "clean_chars").alias("end"),
                    F.greatest(F.lit(0), start - REVIEW_CONTEXT_CODEPOINTS).alias("context_start"),
                    F.least(
                        start + REVIEW_CORE_CODEPOINTS + REVIEW_CONTEXT_CODEPOINTS, "clean_chars"
                    ).alias("context_end"),
                ),
            ),
        )
    )

    with tempfile.TemporaryDirectory(prefix=".chunk-review-", dir=queue_dir.parent) as local:
        staged_queue = Path(local) / "snapshot"
        (
            queue_source.write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged_queue / "queue"))
        )
        new_queue = spark.read.parquet(str(staged_queue / "queue"))
        coverage = (
            new_queue.agg(
                F.count("*").alias("required_parents"),
                F.countDistinct("release_row").alias("distinct_parents"),
                F.sum(F.size("review_units")).alias("required_work_units"),
                F.count_if(
                    F.aggregate(
                        "review_units", F.lit(0), lambda total, unit: total + unit.end - unit.start
                    )
                    == F.col("clean_chars")
                ).alias("parents_with_full_review_windows"),
                F.count_if(F.col("plan_sha256").isNull()).alias("missing_plan_hashes"),
            )
            .first()
            .asDict()
        )
        expected_scope = review_population.groupBy(F.size("review_reasons") > 0).count().collect()
        population_counts = {row[0]: row["count"] for row in expected_scope}
        parent_count = population_counts.get(True, 0)
        if (
            sum(population_counts.values()) != expected
            or any(
                coverage[k] != parent_count
                for k in (
                    "required_parents",
                    "distinct_parents",
                    "parents_with_full_review_windows",
                )
            )
            or coverage["missing_plan_hashes"]
        ):
            raise ValueError("The review queue does not cover exactly the declared population.")
        by_format = [
            row.asDict()
            for row in new_queue.groupBy("format")
            .agg(
                F.count("*").alias("parents"),
                F.countDistinct("source_id").alias("source_ids"),
                F.sum(F.size("review_units")).alias("work_units"),
            )
            .orderBy("format")
            .collect()
        ]
        reason_counts = [
            row.asDict()
            for row in new_queue.select(F.explode("review_reasons").alias("reason"))
            .groupBy("reason")
            .count()
            .orderBy("reason")
            .collect()
        ]
        queue_audit = {
            "formal_parents": expected,
            "outside_declared_complex_scope": population_counts.get(False, 0),
            **coverage,
            "by_format": by_format,
            "reason_counts": reason_counts,
            "initial_status": "pending_external_or_local_reviewer_decisions",
        }
        queue_audit_path = staged_queue / "AUDIT.json"
        queue_audit_path.write_text(json.dumps(queue_audit, indent=2) + "\n")
        queue_files = [
            {
                "path": p.relative_to(staged_queue).as_posix(),
                "sha256": sha256_file(p),
                "rows": pq.ParquetFile(p).metadata.num_rows,
            }
            for p in sorted((staged_queue / "queue").glob("*.parquet"))
        ]
        queue_manifest = {
            **queue_contract,
            "audit_sha256": sha256_file(queue_audit_path),
            "queue_files": queue_files,
        }
        (staged_queue / "MANIFEST.json").write_text(json.dumps(queue_manifest, indent=2) + "\n")
        bound_review_table(staged_queue, "queue", queue_files)
        if queue_dir.exists():
            raise FileExistsError("The review queue destination appeared during publication.")
        staged_queue.rename(queue_dir)
    review_queue = spark.read.parquet(str(queue_dir / "queue"))

display(pd.DataFrame(queue_audit["by_format"]))
display(pd.DataFrame(queue_audit["reason_counts"]))
print(
    f"{queue_audit['required_parents']:,} parents require agent review in "
    f"{queue_audit['required_work_units']:,} work units."
)

## Verify the review packet and decision contracts

An agent reads each unit with its surrounding context and returns the packet hash,
reviewer and revision, outcome, reason and selected boundary IDs. Reviewers can accept the
baseline boundaries, adjust them using candidate IDs, or escalate. Candidate IDs refer to
positions in the immutable parent text. Inspect whether attribution, negation, quotation,
intensity and temperature references stay connected across each proposed boundary.

A parent is ready under the boundary review protocol after every current unit has one
accepted or adjusted decision and the offsets form a complete ordered plan. Missing or
escalated units remain open. A calibration batch can estimate effort and improve the
instructions. Completion is measured on the full declared queue. Token limits and cross-split
checks apply separately to an adopted plan.


In [ ]:
from pyspark.sql import Row

SENTENCE_END = re.compile(r"""[.!?。！？]+["'”’\)\]]{0,3}[ \t\n]+""")


def canonical_sha256(value):
    serialized = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def make_review_packets(queued_parent, body, paragraphs):
    parent_sha = hashlib.sha256(body.encode("utf-8")).hexdigest()
    if parent_sha != queued_parent.clean_sha256 or len(body) != queued_parent.clean_chars:
        raise ValueError("The review body does not match the queued parent.")
    baseline = queued_parent.plan.ends
    if baseline != sorted(set(baseline)) or baseline[-1] != len(body):
        raise ValueError("The parent baseline does not cover the body.")
    proposed = set(baseline[:-1])
    candidates = proposed | {p.start for p in paragraphs if p.start > 0}
    candidates |= {
        match.end() for match in SENTENCE_END.finditer(body) if 0 < match.end() < len(body)
    }
    packet_list = []
    cursor = 0
    for ordinal, unit in enumerate(queued_parent.review_units):
        if not (
            unit.unit_ordinal == ordinal
            and unit.start == cursor
            and 0 <= unit.context_start <= unit.start < unit.end <= unit.context_end <= len(body)
        ):
            raise ValueError("The review windows do not tile the parent body.")
        available = sorted(offset for offset in candidates if unit.start <= offset < unit.end)
        packet = {
            "review_policy": REVIEW_POLICY,
            "release_row": queued_parent.release_row,
            "body_sha256": parent_sha,
            "plan_sha256": queued_parent.plan_sha256,
            "unit_ordinal": ordinal,
            "start": unit.start,
            "end": unit.end,
            "context_start": unit.context_start,
            "context_end": unit.context_end,
            "baseline_boundary_ids": [offset for offset in available if offset in proposed],
            "candidate_boundary_ids": available,
            "context_text": body[unit.context_start : unit.context_end],
        }
        packet_list.append({**packet, "packet_sha256": canonical_sha256(packet)})
        cursor = unit.end
    if cursor != len(body):
        raise ValueError("The review packets do not cover the complete body.")
    return packet_list


def validate_reviewed_parent(queued_parent, body, paragraphs, decisions):
    packets = make_review_packets(queued_parent, body, paragraphs)
    by_hash = {packet["packet_sha256"]: packet for packet in packets}
    supplied = {}
    accepted_boundaries = []
    escalated = False
    for decision in decisions:
        digest = decision["packet_sha256"]
        if digest not in by_hash or digest in supplied:
            raise ValueError("The review decision refers to a stale or duplicate packet.")
        status = decision["status"]
        if status not in {"accept", "adjust", "escalate"} or any(
            not isinstance(decision.get(key), str) or not decision[key].strip()
            for key in ("reviewer", "reviewer_version", "reason")
        ):
            raise ValueError("The review decision is missing an outcome or provenance.")
        packet = by_hash[digest]
        ids = decision["selected_boundary_ids"]
        if ids != sorted(set(ids)) or not set(ids) <= set(packet["candidate_boundary_ids"]):
            raise ValueError("The review decision selects an unavailable boundary.")
        if status == "accept" and ids != packet["baseline_boundary_ids"]:
            raise ValueError("An accepted packet must preserve the proposed boundaries.")
        if status == "adjust" and ids == packet["baseline_boundary_ids"]:
            raise ValueError("An adjusted packet must change at least one proposed boundary.")
        if status == "escalate":
            if ids:
                raise ValueError("Escalate the packet before committing boundary choices.")
            escalated = True
        accepted_boundaries.extend(ids)
        supplied[digest] = status
    if escalated or len(supplied) != len(by_hash):
        return {
            "status": "escalated" if escalated else "pending",
            "required_units": len(by_hash),
            "received_units": len(supplied),
            "ends": None,
        }
    selected_ends = sorted(accepted_boundaries) + [len(body)]
    if selected_ends != sorted(set(selected_ends)) or not all(
        0 < end <= len(body) for end in selected_ends
    ):
        raise ValueError("The assembled reviewed plan must have distinct ordered boundaries.")
    return {
        "status": "boundary_review_complete",
        "required_units": len(by_hash),
        "received_units": len(supplied),
        "ends": selected_ends,
    }


# A UTF-8 body split into two review units exercises complete and incomplete acceptance.
example = "Not afraid at 40°C. Next sentence.\n\nAnother speaker worried."
fixture = Row(
    release_row=0,
    clean_sha256=hashlib.sha256(example.encode()).hexdigest(),
    clean_chars=len(example),
    plan=Row(ends=[20, len(example)]),
    plan_sha256="fixture-plan",
    review_units=[
        Row(unit_ordinal=0, start=0, end=28, context_start=0, context_end=40),
        Row(unit_ordinal=1, start=28, end=len(example), context_start=14, context_end=len(example)),
    ],
)
fixture_paragraphs = [Row(start=0), Row(start=36)]
fixture_packets = make_review_packets(fixture, example, fixture_paragraphs)
fixture_decisions = [
    {
        "packet_sha256": packet["packet_sha256"],
        "status": "accept",
        "reviewer": "fixture",
        "reviewer_version": "example_v1",
        "reason": "The proposed boundary preserves the evidence in the context.",
        "selected_boundary_ids": packet["baseline_boundary_ids"],
    }
    for packet in fixture_packets
]
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, fixture_decisions)[
    "ends"
] == (fixture.plan.ends)
assert (
    validate_reviewed_parent(fixture, example, fixture_paragraphs, fixture_decisions[:1])["status"]
    == "pending"
)
adjusted = [dict(row) for row in fixture_decisions]
adjusted[1].update(status="adjust", selected_boundary_ids=[36])
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, adjusted)["ends"] == [
    20,
    36,
    len(example),
]
escalated = [dict(row) for row in fixture_decisions]
escalated[1]["status"] = "escalate"
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, escalated)["ends"] is None
for bad in (
    [*fixture_decisions, fixture_decisions[0]],
    [{**fixture_decisions[0], "packet_sha256": "wrong-packet"}, fixture_decisions[1]],
    [{**fixture_decisions[0], "selected_boundary_ids": [-1]}, fixture_decisions[1]],
):
    try:
        validate_reviewed_parent(fixture, example, fixture_paragraphs, bad)
    except ValueError:
        pass
    else:
        raise AssertionError("An invalid agent decision was accepted.")

probe = review_queue.where(F.size("review_units") > 1).orderBy("release_row").first()
if probe is not None:
    text_row = (
        parents.where(F.col("release_row") == probe.release_row)
        .select("clean_text", "paragraphs")
        .first()
    )
    packet_probe = make_review_packets(probe, text_row.clean_text, text_row.paragraphs)
    assert len(packet_probe) == len(probe.review_units)
    assert packet_probe[-1]["end"] == probe.clean_chars
print("Verified the review protocol on synthetic decisions and one real packet set.")
print("The published queue starts with every required work unit awaiting a reviewer decision.")

## Review split boundaries

Use the reference candidate for routine records. Assign the complex cases in small,
complete-parent batches to Codex desktop windows. Each window views the same
versioned packet file and writes its own receipt. The coordinating window checks
the assigned packet hashes, evidence continuity, coverage and token budget after
each batch. Keep pending and escalated cases visible. Check exact cross-split reuse
again on the adopted boundaries.

The initial batch measures elapsed time, revisions, evidence kept together and
escalations before estimating a larger run.

## Build a full reference-token boundary candidate

Build a lossless planning candidate for all 718,330 verified parents. Use 512 content
tokens under the pinned `cl100k_base` reference encoding. Use paragraph starts first,
then plausible sentence breaks, whitespace and Unicode grapheme boundaries as needed.
The planner checks the actual encoding of every accepted slice. It retains each parent,
its source hash and inherited split.

The reference budget is an evaluation setting. The tokenizer and any special-token
overhead for an adopted downstream model must be checked separately. This snapshot
stores offsets and counts. It reports exact cross-split chunk reuse and preserves every
known overlap flag for review.

Mother-document metadata, lineage and grouping precede this stage. Evidence spans,
human labels and model-specific features can be joined to verified chunks after their
boundaries are adopted. Use these candidate offsets as inputs for Codex window review.


In [ ]:
from bisect import bisect_left, bisect_right

import regex
import tiktoken

REFERENCE_ENCODING = "cl100k_base"
REFERENCE_CAP = 512
REFERENCE_VOCAB_SHA256 = "223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7"
TOKEN_PLAN_VERSION = "reference_boundary_packing_v1"
SENTENCE_BREAK = regex.compile(r"""[.!?。！？]+["'”’\)\]]{0,3}\s+""")
WHITESPACE_BREAK = regex.compile(r"\s+")
GRAPHEME = regex.compile(r"\X")


def reference_plan(text, paragraphs, limit=REFERENCE_CAP):
    encoder = tiktoken.get_encoding(REFERENCE_ENCODING)
    size = len(text)
    ids = encoder.encode_ordinary(text)
    if size == 0:
        return [], [], []
    if len(ids) <= limit:
        return [size], [len(ids)], ["terminal"]

    decoded, offsets = encoder.decode_with_offsets(ids)
    if decoded != text:
        raise ValueError("The reference tokenizer did not round-trip the parent text.")
    offsets.append(size)
    anchors = (
        ("paragraph", [span.start for span in paragraphs]),
        ("sentence", [m.end() for m in SENTENCE_BREAK.finditer(text)]),
        ("whitespace", [m.end() for m in WHITESPACE_BREAK.finditer(text)]),
    )
    graphemes = None
    ends, counts, kinds = [], [], []
    start = 0
    while start < size:
        token_start = bisect_left(offsets, start)
        upper = max(start + 1, offsets[min(token_start + limit, len(ids))])
        while True:
            stop, kind = upper, "terminal"
            if upper < size:
                minimum = start + max(1, (upper - start) // 3)
                for name, positions in anchors:
                    index = bisect_right(positions, upper) - 1
                    if index >= 0 and positions[index] >= minimum:
                        stop, kind = positions[index], name
                        break
                else:
                    if graphemes is None:
                        graphemes = [m.end() for m in GRAPHEME.finditer(text)]
                    index = bisect_right(graphemes, upper) - 1
                    if index >= 0 and graphemes[index] > start:
                        stop, kind = graphemes[index], "grapheme"
                    else:
                        kind = "codepoint"
            count = len(encoder.encode_ordinary(text[start:stop]))
            if 0 < count <= limit:
                ends.append(stop)
                counts.append(count)
                kinds.append(kind)
                start = stop
                break
            if stop <= start + 1:
                raise ValueError("A single code point exceeds the reference token budget.")
            upper = max(start + 1, min(stop - 1, start + (stop - start) * limit // count))
    return ends, counts, kinds


def assert_reference_plan(text, plans, budget):
    codec = tiktoken.get_encoding(REFERENCE_ENCODING)
    ends, counts, kinds = plans
    assert len(ends) == len(counts) == len(kinds)
    assert ends[-1:] == ([len(text)] if text else [])
    reconstructed = []
    cursor = 0
    for end, expected in zip(ends, counts, strict=True):
        assert cursor < end <= len(text)
        raw = text[cursor:end]
        assert len(codec.encode_ordinary(raw)) == expected <= budget
        reconstructed.append(raw)
        cursor = end
    assert "".join(reconstructed) == text


examples = (
    ("Not afraid of summer. Arvo at 40°C. 👩🏽‍🚒", [], 512),
    (("Rain eased. The speaker was not afraid. " * 40).rstrip(), [], 32),
    ("alpha\n\n" + "not afraid but anxious. " * 70, [], 33),
    ("👨🏾‍👩‍👧‍👦" * 100, [], 79),
    ("字" * 210, [], 32),
    ("abc" + "\u0301" * 500 + " word", [], 32),
    ("x" * 10000, [], 64),
    ("", [], 512),
)
for body, spans, cap in examples:
    assert_reference_plan(body, reference_plan(body, spans, cap), cap)
assert reference_plan(examples[0][0], [], REFERENCE_CAP)[2] == ["terminal"]
print(f"{len(examples)} reference packing and round-trip cases passed.")

In [ ]:
from hashlib import sha1
from importlib.metadata import version

import nbformat

REFERENCE_VOCAB_URL = "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken"
cache_setting = os.environ.get("TIKTOKEN_CACHE_DIR", os.environ.get("DATA_GYM_CACHE_DIR"))
vocabulary_cache = (
    Path(cache_setting) if cache_setting else Path(tempfile.gettempdir()) / "data-gym-cache"
)
vocabulary_file = vocabulary_cache / sha1(REFERENCE_VOCAB_URL.encode()).hexdigest()
if not vocabulary_file.is_file() or sha256_file(vocabulary_file) != REFERENCE_VOCAB_SHA256:
    raise ValueError("Provision the pinned cl100k_base vocabulary in the local tokenizer cache.")
notebook_file = Path(
    os.environ.get("FOT06_SOURCE_NOTEBOOK", ROOT / "notebooks/06_chunk_evaluation.ipynb")
)
saved_code = nbformat.read(notebook_file, as_version=4)
policy = next(cell.source for cell in saved_code.cells if cell.id == "06-token-policy")
policy_sha = hashlib.sha256(policy.encode("utf-8")).hexdigest()
token_dir = HANDOFF.with_name(f"{HANDOFF.name}_reference_chunks_v1")
token_identity = {
    "schema_version": 1,
    "policy": TOKEN_PLAN_VERSION,
    "policy_source_sha256": policy_sha,
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
    "encoding": REFERENCE_ENCODING,
    "vocabulary_sha256": REFERENCE_VOCAB_SHA256,
    "tokenizer_version": version("tiktoken"),
    "grapheme_regex_version": version("regex"),
    "reference_content_budget": REFERENCE_CAP,
}

if token_dir.exists():
    token_manifest = json.loads((token_dir / "MANIFEST.json").read_text())
    if any(token_manifest.get(key) != value for key, value in token_identity.items()):
        raise ValueError("The saved tokenizer candidate belongs to a different input or policy.")
    receipt = token_dir / "AUDIT.json"
    if sha256_file(receipt) != token_manifest["audit_sha256"]:
        raise ValueError("The tokenizer candidate audit failed verification.")
    token_plans = bound_review_table(token_dir, "plans", token_manifest["files"])
    token_audit = json.loads(receipt.read_text())
else:
    packing_udf = F.udf(
        reference_plan,
        "struct<ends:array<int>,counts:array<int>,boundary_kinds:array<string>>",
        useArrow=False,
    )
    with tempfile.TemporaryDirectory(prefix=".reference-chunks-", dir=HANDOFF.parent) as temp:
        stage = Path(temp) / "snapshot"
        (
            parents.select(
                "release_row",
                "native_identity",
                "release_group_id",
                "ml_split",
                "format",
                "clean_sha256",
                "clean_chars",
                "cross_split_review_required",
                packing_udf("clean_text", "paragraphs").alias("plan"),
            )
            .write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(stage / "plans"))
        )
        measured = spark.read.parquet(str(stage / "plans"))
        previous = F.slice(F.concat(F.array(F.lit(0)), "plan.ends"), 1, F.size("plan.ends"))
        valid = (
            (F.size("plan.ends") > 0)
            & (F.size("plan.ends") == F.size("plan.counts"))
            & (F.size("plan.ends") == F.size("plan.boundary_kinds"))
            & (F.try_element_at("plan.ends", F.lit(-1)) == F.col("clean_chars"))
            & F.forall(
                F.zip_with("plan.ends", previous, lambda end, start: end > start),
                lambda in_order: in_order,
            )
            & F.forall("plan.counts", lambda n: n.between(1, REFERENCE_CAP))
        )
        tests = (
            measured.agg(
                F.count("*").alias("parents"),
                F.countDistinct("release_row").alias("unique_parents"),
                F.count_if(F.coalesce(valid, F.lit(False))).alias("valid_and_covered_parents"),
                F.count_if(F.size("plan.ends") == 1).alias("single_chunk_parents"),
                F.sum(F.size("plan.ends")).alias("chunks"),
                F.sum(F.aggregate("plan.counts", F.lit(0), lambda total, n: total + n)).alias(
                    "reference_tokens"
                ),
                F.max(F.array_max("plan.counts")).alias("largest_chunk_tokens"),
            )
            .first()
            .asDict()
        )
        if any(
            tests[key] != expected
            for key in ("parents", "unique_parents", "valid_and_covered_parents")
        ):
            raise ValueError(f"The token plan did not cover the complete corpus: {tests}")

        split_counts = {
            row.ml_split: row["count"] for row in measured.groupBy("ml_split").count().collect()
        }
        source_splits = {}
        for row in closure["overlap_status_by_split"]:
            name = row["ml_split"]
            source_splits[name] = source_splits.get(name, 0) + row["count"]
        if split_counts != source_splits:
            raise ValueError("Chunk planning changed the inherited split roster.")

        by_format = [
            row.asDict()
            for row in measured.groupBy("format")
            .agg(
                F.count("*").alias("parents"),
                F.sum(F.size("plan.ends")).alias("chunks"),
                F.count_if(F.size("plan.ends") == 1).alias("single_chunk_parents"),
            )
            .orderBy("format")
            .collect()
        ]
        by_kind = [
            row.asDict()
            for row in measured.select(F.explode("plan.boundary_kinds").alias("boundary_kind"))
            .groupBy("boundary_kind")
            .count()
            .orderBy("boundary_kind")
            .collect()
        ]
        chunk_positions = measured.select(
            "release_row",
            "clean_sha256",
            "ml_split",
            "cross_split_review_required",
            "plan.ends",
            F.posexplode("plan.ends").alias("ordinal", "end"),
        ).withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
        chunk_hashes = (
            chunk_positions.join(
                parents.select("release_row", "clean_sha256", "clean_text"),
                ["release_row", "clean_sha256"],
                "inner",
            )
            .withColumn("chars", F.col("end") - F.col("start"))
            .select(
                "ml_split",
                "cross_split_review_required",
                "chars",
                F.sha2(F.substring("clean_text", F.col("start") + 1, F.col("chars")), 256).alias(
                    "chunk_sha256"
                ),
            )
        )
        split_bit = (
            F.when(F.col("ml_split") == "train", 1)
            .when(F.col("ml_split") == "validation", 2)
            .otherwise(4)
        )
        comparison = ~F.col("cross_split_review_required")
        reuse = (
            chunk_hashes.groupBy("chunk_sha256", "chars")
            .agg(
                F.bit_or(split_bit).alias("all_splits"),
                F.bit_or(F.when(comparison, split_bit).otherwise(0)).alias("comparison_splits"),
                F.count("*").alias("occurrences"),
                F.count_if(comparison).alias("comparison_occurrences"),
            )
            .where(F.col("all_splits").isin(3, 5, 6, 7))
        )
        reuse_stats = (
            reuse.agg(
                F.count("*").alias("cross_split_hashes"),
                F.coalesce(F.sum("occurrences"), F.lit(0)).alias("cross_split_occurrences"),
                F.count_if(F.col("comparison_splits").isin(3, 5, 6, 7)).alias(
                    "cross_split_hashes_without_known_parent_overlap"
                ),
                F.count_if(F.col("chars") >= 500).alias(
                    "cross_split_hashes_at_least_500_codepoints"
                ),
                F.count_if(
                    (F.col("chars") >= 500) & F.col("comparison_splits").isin(3, 5, 6, 7)
                ).alias("long_cross_split_hashes_without_known_parent_overlap"),
            )
            .first()
            .asDict()
        )
        token_audit = {
            **tests,
            "split_counts": split_counts,
            "by_format": by_format,
            "by_boundary_kind": by_kind,
            "exact_chunk_reuse": reuse_stats,
            "status": "full_reference_candidate_ready_for_boundary_review",
        }
        files = [
            {
                "path": part.relative_to(stage).as_posix(),
                "rows": pq.ParquetFile(part).metadata.num_rows,
                "sha256": sha256_file(part),
            }
            for part in sorted((stage / "plans").glob("*.parquet"))
        ]
        receipt = stage / "AUDIT.json"
        receipt.write_text(json.dumps(token_audit, indent=2) + "\n")
        token_manifest = {
            **token_identity,
            "spark_master": spark.sparkContext.master,
            "files": files,
            "audit_sha256": sha256_file(receipt),
        }
        (stage / "MANIFEST.json").write_text(json.dumps(token_manifest, indent=2) + "\n")
        bound_review_table(stage, "plans", files)
        if token_dir.exists():
            raise FileExistsError(
                "The tokenizer candidate destination appeared during publication."
            )
        stage.rename(token_dir)
    token_plans = spark.read.parquet(str(token_dir / "plans"))

print(f"Verified {token_audit['valid_and_covered_parents']:,} reference-token plans.")
display(pd.DataFrame(token_audit["by_format"]))
display(pd.DataFrame(token_audit["by_boundary_kind"]))
display(pd.DataFrame(token_audit["exact_chunk_reuse"].items(), columns=["Measure", "Count"]))

In [ ]:
def validate_reference_decisions(queued_parent, body, paragraphs, decisions):
    result = validate_reviewed_parent(queued_parent, body, paragraphs, decisions)
    ends = result["ends"]
    if ends is None:
        return result
    encoder = tiktoken.get_encoding(REFERENCE_ENCODING)
    counts = [
        len(encoder.encode_ordinary(body[start:end]))
        for start, end in zip([0, *ends[:-1]], ends, strict=True)
    ]
    oversized = [ordinal for ordinal, count in enumerate(counts) if count > REFERENCE_CAP]
    if oversized:
        return {
            **result,
            "status": "reference_budget_review_required",
            "ends": None,
            "over_budget_chunk_ordinals": oversized,
        }
    return {**result, "reference_token_counts": counts}


gate_body = ("The speaker said she was not afraid of the heat. " * 150).rstrip()
gate_baseline = reference_plan(gate_body, [Row(start=0)])
gate_parent = Row(
    release_row=0,
    clean_sha256=hashlib.sha256(gate_body.encode()).hexdigest(),
    clean_chars=len(gate_body),
    plan=Row(ends=gate_baseline[0]),
    plan_sha256="reference-budget-fixture",
    review_units=[
        Row(
            unit_ordinal=0, start=0, end=len(gate_body), context_start=0, context_end=len(gate_body)
        ),
    ],
)
packet = make_review_packets(gate_parent, gate_body, [Row(start=0)])[0]
fixture_receipt = {
    "packet_sha256": packet["packet_sha256"],
    "status": "accept",
    "reviewer": "fixture",
    "reviewer_version": "reference_fixture_v1",
    "reason": "Retain the proposed boundaries.",
    "selected_boundary_ids": packet["baseline_boundary_ids"],
}
good = validate_reference_decisions(gate_parent, gate_body, [Row(start=0)], [fixture_receipt])
assert good["ends"] == gate_baseline[0] and good["reference_token_counts"] == gate_baseline[1]
merged = validate_reference_decisions(
    gate_parent,
    gate_body,
    [Row(start=0)],
    [
        {
            **fixture_receipt,
            "status": "adjust",
            "selected_boundary_ids": [],
            "reason": "Merge the proposed chunks for the budget test.",
        }
    ],
)
assert merged["status"] == "reference_budget_review_required" and merged["ends"] is None
print("Verified the reference plan and held a reviewer adjustment that exceeded its token budget.")

## Prepare the first Codex window assignment

Freeze 240 development units using complete parents and the existing selection seed.
Rebind each unit to the local reference plan. Publish one read-only packet file with
a separate line index and a first assignment of up to 12 units. The coordinating window
dispatches the assignment using the shared workspace. Future windows can receive a
different parent roster in the same file.

See [Codex window handoff](PREPROCESSING_WINDOW_HANDOFF.md) for packet offsets, receipt
fields and how to resume. Opening the notebook prepares and validates local files.

In [ ]:
from pyspark.sql import Window

# Retain the previous draw salt to reproduce the frozen cohort.
PILOT_SEED = "paired-medium-high-pilot-v1"
PILOT_TARGET_UNITS = 240
pilot_dir = queue_dir.with_name(f"{queue_dir.name}_window_sample_v1")
pilot_contract = {
    "draw_seed": PILOT_SEED,
    "review_route": "codex_desktop_window",
    "queue_manifest_sha256": sha256_file(queue_dir / "MANIFEST.json"),
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
    "near_delta_manifest_sha256": sha256_file(near_manifest_path),
    "target_units": PILOT_TARGET_UNITS,
}
pilot_manifest_path = pilot_dir / "MANIFEST.json"
if pilot_dir.exists():
    pilot_manifest = json.loads(pilot_manifest_path.read_text())
    if any(pilot_manifest.get(key) != value for key, value in pilot_contract.items()):
        raise ValueError("The saved pilot belongs to a different queue or selection policy.")
    if sha256_file(pilot_dir / "PACKETS.json") != pilot_manifest["packets_sha256"]:
        raise ValueError("The pilot packet inventory failed verification.")
    pilot_inventory = json.loads((pilot_dir / "PACKETS.json").read_text())
else:
    review_evidence = F.exists(
        "review_reasons",
        lambda reason: reason.isin("retained_markup_or_spacing", "upstream_review_evidence"),
    )
    eligible = (
        review_queue.where(
            (F.col("ml_split") == "train")
            & (F.col("overlap_status") == "no_cross_split_evidence_in_current_audit")
            & ~F.array_contains("review_reasons", "incremental_near_duplicate_match")
        )
        .withColumn("units", F.size("review_units"))
        .withColumn(
            "tier",
            F.when(F.col("units") > 1, "multiple_windows")
            .when(review_evidence, "structural_review")
            .when(F.col("clean_chars") <= 1200, "short_control")
            .otherwise("longer_single_window"),
        )
        .withColumn(
            "draw",
            F.sha2(
                F.concat_ws("|", F.lit(PILOT_SEED), "native_identity", "release_group_id"),
                256,
            ),
        )
    )
    group_draw = Window.partitionBy("release_group_id").orderBy("draw", "release_row")
    source_draw = Window.partitionBy("tier", "format", "source_id").orderBy("draw", "release_row")
    format_draw = Window.partitionBy("tier", "format").orderBy("source_rank", "draw", "release_row")
    pool = (
        eligible.withColumn("group_rank", F.row_number().over(group_draw))
        .where(F.col("group_rank") == 1)
        .withColumn("source_rank", F.row_number().over(source_draw))
        .withColumn("format_rank", F.row_number().over(format_draw))
        .where(F.col("format_rank") <= 100)
        .select(
            "release_row",
            "release_group_id",
            "format",
            "source_id",
            "tier",
            "units",
            "draw",
            "format_rank",
        )
        .orderBy("format_rank", "draw", "release_row")
        .collect()
    )
    budgets = {
        "multiple_windows": 96,
        "structural_review": 48,
        "longer_single_window": 48,
        "short_control": 48,
    }
    picked = {}
    by_tier = {tier: 0 for tier in budgets}
    for tier, quota in budgets.items():
        for row in pool:
            if row.tier == tier and row.units <= quota - by_tier[tier]:
                picked[row.release_row] = row
                by_tier[tier] += row.units
                if by_tier[tier] == quota:
                    break
    remaining = PILOT_TARGET_UNITS - sum(by_tier.values())
    for row in pool:
        if remaining == 0:
            break
        if row.tier == "short_control" and row.release_row not in picked:
            picked[row.release_row] = row
            by_tier[row.tier] += row.units
            remaining -= row.units
    if remaining or len({row.release_group_id for row in picked.values()}) != len(picked):
        raise ValueError("The pilot must cover 240 units in distinct complete parent groups.")

    pilot_ids = F.col("release_row").isin(*picked)
    queued_pilot = {row.release_row: row for row in review_queue.where(pilot_ids).collect()}
    text_rows = parents.where(pilot_ids).select("release_row", "clean_text", "paragraphs").collect()
    if set(queued_pilot) != set(picked) or {row.release_row for row in text_rows} != set(picked):
        raise ValueError("A selected parent is missing from the text or review queue.")

    PACKET_INDEX_FIELDS = ("packet_sha256", "unit_ordinal", "start", "end")
    pilot_inventory = []
    for row in sorted(text_rows, key=lambda row: row.release_row):
        queued = queued_pilot[row.release_row]
        packets = make_review_packets(queued, row.clean_text, row.paragraphs)
        for packet in packets:
            pilot_inventory.append(
                {
                    "release_row": row.release_row,
                    "release_group_id": queued.release_group_id,
                    "format": queued.format,
                    "source_id": queued.source_id,
                    "tier": picked[row.release_row].tier,
                    "body_sha256": queued.clean_sha256,
                    "plan_sha256": queued.plan_sha256,
                    **{key: packet[key] for key in PACKET_INDEX_FIELDS},
                }
            )
    if (
        len(pilot_inventory) != PILOT_TARGET_UNITS
        or len({packet["packet_sha256"] for packet in pilot_inventory}) != PILOT_TARGET_UNITS
    ):
        raise ValueError("The pilot inventory contains a missing or repeated packet.")
    with tempfile.TemporaryDirectory(prefix=".review-pilot-", dir=pilot_dir.parent) as local:
        staged_pilot = Path(local) / "snapshot"
        staged_pilot.mkdir()
        file = staged_pilot / "PACKETS.json"
        file.write_text(json.dumps(pilot_inventory, indent=2) + "\n")
        pilot_manifest = {
            **pilot_contract,
            "full_parent_records": len(picked),
            "source_ids": len({row.source_id for row in picked.values()}),
            "format_count": len({row.format for row in picked.values()}),
            "units_by_tier": by_tier,
            "packets_sha256": sha256_file(file),
        }
        (staged_pilot / "MANIFEST.json").write_text(json.dumps(pilot_manifest, indent=2) + "\n")
        if pilot_dir.exists():
            raise FileExistsError("The pilot destination appeared during publication.")
        staged_pilot.rename(pilot_dir)

print(
    f"Frozen {len(pilot_inventory)} review units across {pilot_manifest['full_parent_records']} "
    f"complete parents and {pilot_manifest['source_ids']} sources."
)
display(pd.DataFrame(pilot_manifest["units_by_tier"].items(), columns=["Tier", "Units"]))

In [ ]:
WINDOW_POLICY = "codex-window-review-v1"
reference_manifest_sha = sha256_file(token_dir / "MANIFEST.json")
window_identity = {
    "policy": WINDOW_POLICY,
    "reference_plan_manifest_sha256": reference_manifest_sha,
    "selection_manifest_sha256": sha256_file(pilot_manifest_path),
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
}
window_root = ROOT / ".cache" / "chunk-window-review"
window_root.mkdir(mode=0o700, parents=True, exist_ok=True)
window_dir = window_root / canonical_sha256(window_identity)[:20]
window_manifest_path = window_dir / "MANIFEST.json"


def reference_review_parent(queued_parent, reference_plan_row):
    if queued_parent.clean_sha256 != reference_plan_row.clean_sha256:
        raise ValueError("The reference plan belongs to a different review body.")
    ends = reference_plan_row.plan.ends
    return Row(
        **{
            **queued_parent.asDict(),
            "plan": Row(ends=ends),
            "plan_sha256": canonical_sha256(
                {
                    "reference_manifest_sha256": reference_manifest_sha,
                    "body_sha256": reference_plan_row.clean_sha256,
                    "ends": ends,
                }
            ),
        }
    )


if not window_dir.exists():
    selected = {item["release_row"] for item in pilot_inventory}
    chosen = F.col("release_row").isin(*sorted(selected))
    queued = {row.release_row: row for row in review_queue.where(chosen).collect()}
    packed = {
        row.release_row: row
        for row in token_plans.where(chosen).select("release_row", "clean_sha256", "plan").collect()
    }
    corpus = parents.where(chosen).select("release_row", "clean_text", "paragraphs").collect()
    if (
        {row.release_row for row in corpus} != selected
        or set(queued) != selected
        or set(packed) != selected
    ):
        raise ValueError("The reference plans do not cover the frozen review parents.")

    original = {(item["release_row"], item["unit_ordinal"]): item for item in pilot_inventory}
    packets = []
    index = []
    for row in sorted(corpus, key=lambda row: row.release_row):
        earlier = queued[row.release_row]
        reference = packed[row.release_row]
        reference_parent = reference_review_parent(earlier, reference)
        for packet in make_review_packets(reference_parent, row.clean_text, row.paragraphs):
            old = original.pop((row.release_row, packet["unit_ordinal"]))
            if (
                (packet["start"], packet["end"]) != (old["start"], old["end"])
                or packet["packet_sha256"] == old["packet_sha256"]
            ):
                raise ValueError("A reference packet changed range or reused the old hash.")
            packets.append(packet)
            index.append(
                {
                    "line": len(packets),
                    "release_row": row.release_row,
                    "tier": old["tier"],
                    "format": old["format"],
                    "packet_sha256": packet["packet_sha256"],
                }
            )
    if original or len(packets) != PILOT_TARGET_UNITS:
        raise ValueError("The reference review must include every frozen pilot window.")

    by_parent = {}
    for item in index:
        by_parent.setdefault(item["release_row"], []).append(item)
    # Give the first window complex cases and a short control, keeping each parent intact.
    first_batch = []
    for tier, quota in (("multiple_windows", 8), ("structural_review", 3), ("short_control", 1)):
        remaining = quota
        for units in by_parent.values():
            if units[0]["tier"] == tier and len(units) <= remaining:
                first_batch.extend(units)
                remaining -= len(units)
                if not remaining:
                    break
    if not first_batch:
        raise ValueError("The pilot has no complete parent that fits the first review batch.")

    with tempfile.TemporaryDirectory(prefix=".codex-window-", dir=window_root) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        shared = stage / "PACKETS.jsonl"
        with shared.open("x", encoding="utf-8") as handle:
            for packet in packets:
                handle.write(json.dumps(packet, ensure_ascii=False) + "\n")
        index_file = stage / "INDEX.json"
        index_file.write_text(json.dumps(index, indent=2) + "\n", encoding="utf-8")
        assignment = {
            "batch": "batch-001",
            "source_file": shared.name,
            "source_sha256": sha256_file(shared),
            "units": sorted(first_batch, key=lambda item: item["line"]),
            "receipt": "decisions/batch-001.jsonl",
        }
        assignment_file = stage / "BATCH_001.json"
        assignment_file.write_text(json.dumps(assignment, indent=2) + "\n", encoding="utf-8")
        window_manifest = {
            **window_identity,
            "units": len(packets),
            "parents": len(selected),
            "files": {
                file.name: sha256_file(file) for file in (shared, index_file, assignment_file)
            },
        }
        (stage / "MANIFEST.json").write_text(
            json.dumps(window_manifest, indent=2) + "\n", encoding="utf-8"
        )
        for file in stage.iterdir():
            if file.is_file():
                file.chmod(0o400)
        if window_dir.exists():
            raise FileExistsError("The window review directory appeared during preparation.")
        stage.rename(window_dir)

window_manifest = json.loads(window_manifest_path.read_text(encoding="utf-8"))
if any(window_manifest.get(key) != value for key, value in window_identity.items()):
    raise ValueError("The saved window bundle belongs to a different input or reference plan.")
expected_files = {"PACKETS.jsonl", "INDEX.json", "BATCH_001.json"}
if set(window_manifest["files"]) != expected_files or any(
    sha256_file(window_dir / name) != digest for name, digest in window_manifest["files"].items()
):
    raise ValueError("The shared window bundle failed verification.")

index = json.loads((window_dir / "INDEX.json").read_text(encoding="utf-8"))
assignment = json.loads((window_dir / "BATCH_001.json").read_text(encoding="utf-8"))
assigned_lines = {item["line"] for item in assignment["units"]}
assigned_parents = {item["release_row"] for item in assignment["units"]}
assigned_packets = {item["packet_sha256"] for item in assignment["units"]}
if (
    not 0 < len(assigned_lines) == len(assigned_packets) == len(assignment["units"]) <= 12
    or len(index) != window_manifest["units"]
    or index != sorted(index, key=lambda item: item["line"])
    or assignment["units"] != [item for item in index if item["release_row"] in assigned_parents]
    or assignment["source_file"] != "PACKETS.jsonl"
    or assignment["source_sha256"] != window_manifest["files"]["PACKETS.jsonl"]
    or assignment["receipt"] != "decisions/batch-001.jsonl"
):
    raise ValueError("The initial assignment is not a complete, source-bound parent batch.")

print(
    f"Verified one local source file with {window_manifest['units']} units "
    f"from {window_manifest['parents']} complete parents."
)
print(
    f"Initial Codex assignment: {len(assigned_lines)} units "
    f"from {len(assigned_parents)} parents."
)
print(f"Window handoff directory: {window_dir.relative_to(ROOT)}")


## Validate a returned assignment

Verify the exact packet roster, replay each decision against the original text, and
check each complete parent against the reference token budget. A receipt can contain
escalations. This cell measures review completion for the assigned batch only.

In [ ]:
receipt_path = window_dir / assignment["receipt"]
if receipt_path.is_file():
    with receipt_path.open(encoding="utf-8") as handle:
        decisions = [json.loads(line) for line in handle]
    received = {decision["packet_sha256"]: decision for decision in decisions}
    if len(received) != len(decisions) or set(received) != assigned_packets:
        raise ValueError("The receipt must cover each assigned packet once.")
    chosen = F.col("release_row").isin(*sorted(assigned_parents))
    scoped_queue = {row.release_row: row for row in review_queue.where(chosen).collect()}
    scoped_plans = {
        row.release_row: row
        for row in token_plans.where(chosen).select("release_row", "clean_sha256", "plan").collect()
    }
    results = []
    for row in parents.where(chosen).select("release_row", "clean_text", "paragraphs").collect():
        parent = reference_review_parent(
            scoped_queue[row.release_row], scoped_plans[row.release_row]
        )
        hashes = [
            packet["packet_sha256"]
            for packet in make_review_packets(parent, row.clean_text, row.paragraphs)
        ]
        result = validate_reference_decisions(
            parent, row.clean_text, row.paragraphs, [received[digest] for digest in hashes]
        )
        results.append({"release_row": row.release_row, "status": result["status"]})
    if {row["release_row"] for row in results} != assigned_parents:
        raise ValueError("The validation did not cover the entire assignment.")
    print(f"Verified receipt for {len(received)} review units.")
    print(f"Receipt SHA-256: {sha256_file(receipt_path)}")
    display(pd.DataFrame(decisions).groupby("status").size().rename("review_units").to_frame())
    display(pd.DataFrame(results).groupby("status").size().rename("parents").to_frame())
else:
    print("The first native Codex window receipt is pending.")
